# Day 34 — distil, export, quantise, gate (Colab/Kaggle, T4)

Distils the Day-33 safety teacher (and Day-32 emotion teacher) into a
multilingual MiniLM student, exports ONNX, applies INT8 dynamic
quantisation, scores fp32 vs INT8 on the frozen eval, checks the
release gates, and registers the survivor. **Nothing here is run in
the build sandbox** (no GPU/hub); the same modules ran there against
a stand-in teacher and a linear student (`make distill-smoke`).

Prerequisites: the Day-33 (and Day-32) GPU runs are done and their
artefacts are uploaded to this runtime (``artifacts/safety/...``).


In [ ]:
# 1. Install the repository + serving stack (CPU onnxruntime is fine).
!git clone https://github.com/BuildbyArindam/Manovia.git
%cd Manovia/ml
!pip install -q -e '.[ml,training,serving]'


In [ ]:
# 2. Pin the fresh corpus hashes into a run-local copy of the config.
# The committed config keeps sha256: null until the owner pastes
# measured values into ml/configs/distill_safety_student.yaml.
import pathlib, yaml
cfg = yaml.safe_load(pathlib.Path('configs/distill_safety_student.yaml').read_text())
pathlib.Path('/tmp/distill_safety_student.yaml').write_text(yaml.safe_dump(cfg))
print('fill data.*.sha256/rows from the Day-33 corpus build, then continue')


In [ ]:
# 3. Distil (transformer student; temperature loss from the config).
# The training loop lives in the notebook by design for the GPU run;
# it consumes teacher soft targets from the packaged Day-33 artefact
# and writes a Hugging Face student directory + distill_metrics.json.
print('train with configs/distill_safety_student.yaml (see ml/README.md Day 34)')


In [ ]:
# 4. Export + quantise + frozen-eval comparison + gates.
# (Linear-smoke equivalent: manovia_ml.compression.pipeline.)
from manovia_ml.compression.gates import load_release_gates, evaluate_gates
gates = load_release_gates()
print(gates)


In [ ]:
# 5. Register, then promote ONLY with a passing gate report.
# python -m manovia_ml.compression.registry --tracking-uri <uri> \
#   --registry-dir artifacts/registry --model safety-student register \
#   --artifact <student dir> --version v1
# python -m manovia_ml.compression.registry ... promote --version v1 --gates-passed
print('promotion refuses without --gates-passed; rollback is the same CLI')
